# Hexene epoxidation — NEB coadsorption model

Same plotting and output workflow as the original `test.ipynb`; model species are **R/P/RP**, not OOH/O.

Numerical results use provisional NEB barrier assumptions; do not interpret as validated TOF.


In [ ]:
from pathlib import Path
from catmap import ReactionModel

mkm_file = 'alkene_epoxidation.mkm'
model = ReactionModel(setup_file=mkm_file)
for key in ['production_rate','rate','rate_control','coverage','selectivity_control','rxn_order']:
    if key not in model.output_variables:
        model.output_variables.append(key)
model.run()
print("Reaction model solved; requested output variables:", model.output_variables)


In [ ]:
from catmap import analyze

vm = analyze.VectorMap(model)
vm.plot_variable = 'rate'
vm.log_scale = True
vm.min = 1e-25
vm.max = 1e3
vm.plot(save='rate.pdf')

vm.unique_only = False
vm.plot(save='all_rates.pdf')
vm.unique_only = True

vm.plot_variable = 'production_rate'
vm.production_rate_map = model.production_rate_map
vm.threshold = 1e-30
vm.descriptor_labels = ['R formation free energy [eV]', 'P formation free energy [eV]']
vm.subplots_adjust_kwargs = {'left':0.20, 'right':0.80, 'bottom':0.15}
vm.plot(save='production_rate.pdf')
vm.plot(save='pretty_production_rate.pdf')

vm.plot_variable = 'coverage'
vm.log_scale = False
vm.min = 0
vm.max = 1
vm.include_labels = None
vm.plot(save='coverage.pdf')
vm.include_labels = ['R_s','P_s']
vm.plot(save='R_P_coverage.pdf')

print("Generated PDF plots in:", Path.cwd())


In [ ]:
from catmap import analyze

sa = analyze.ScalingAnalysis(model)
sa.plot(save='scaling.pdf')
print("Generated scaling.pdf")


In [ ]:
from pathlib import Path
import numpy as np

output_variable = 'production_rate'
labels = model.output_labels[output_variable]
map_data = getattr(model,output_variable+'_map')
table = '\t'.join(['descriptor-'+d for d in model.descriptor_names]+list(labels))+'\n'
for pt, output in map_data:
    table += '\t'.join(str(float(v)) for v in list(pt)+list(output))+'\n'
Path(output_variable+'_table.txt').write_text(table, encoding='utf-8')
print('Wrote:', output_variable+'_table.txt')


In [ ]:
labels = model.output_labels['coverage']
for descriptors, cvg in model.coverage_map:
    print('descriptors', descriptors)
    print('intermediates', labels)
    print('coverages', [float(c) for c in cvg])
    break  # remove break to print all grid points
